# Black Friday Sales Prediction

**Tabular Regression Project** — Predict individual purchase amounts from Black Friday transaction data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Black Friday (550,068 rows × 12 columns, sampled to 50,000)
Target: `Purchase`

## 2 · Project Overview

This project focuses on **predicting individual purchase amounts** using customer and product features. Unlike the analysis-focused sibling project, here we emphasize **model optimization**, **feature selection**, and achieving the best predictive performance.

We sample to 50K rows for practical training times while maintaining statistical validity.

## 3 · Learning Objectives

1. Apply systematic feature selection to improve predictions.
2. Optimize gradient-boosting hyperparameters.
3. Compare multiple model families on the same preprocessed data.
4. Use LazyPredict and FLAML for automated model selection.
5. Perform thorough error analysis on retail prediction.

## 4 · Problem Statement

Predict the **Purchase amount** for Black Friday transactions given customer demographics and product category. The focus is on building the most accurate prediction model possible.

## 5 · Why This Project Matters

- Purchase prediction drives **revenue forecasting** and **personalization** in retail.
- Real-world retail data is messy — missing values, mixed types, high cardinality.
- Optimizing for prediction accuracy teaches practical ML workflow.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 550,068 (sampled to 50,000) |
| **Columns** | 12 |
| **Target** | Purchase (USD) |
| **Missing** | Product_Category_2 (~31%), Product_Category_3 (~69%) |
| **File** | `train.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Analytics Vidhya / Kaggle Black Friday dataset.
- **License**: Public / educational use.
- **Note**: Same underlying data as the Analysis project, different modeling focus.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Purchase'
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'train.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Original shape: {df.shape}')

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target stats:')
print(df[TARGET].describe())

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Product_Category_1')[TARGET].mean().plot(kind='bar', ax=axes[0,0], title='Avg Purchase by Product Cat 1')
df.groupby('Marital_Status')[TARGET].mean().plot(kind='bar', ax=axes[0,1], title='Avg Purchase by Marital Status')
df['Stay_In_Current_City_Years'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], title='Stay Distribution')
df[TARGET].hist(bins=50, ax=axes[1,1], edgecolor='black')
axes[1,1].set_title('Purchase Distribution')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(f'Purchase statistics:')
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Kurtosis: {df[TARGET].kurtosis():.2f}')

## 15 · Train / Validation / Test Split Strategy

80/20 random split.

## 16 · Preprocessing Strategy

- Drop identifiers (User_ID, Product_ID)
- Encode categoricals: Gender, Age, City_Category, Stay_In_Current_City_Years
- Fill missing product categories with 0
- Create frequency-based and aggregation features

In [ ]:
# Preprocessing
df = df.drop(columns=['User_ID', 'Product_ID'])

df['Gender'] = df['Gender'].map({'M': 1, 'F': 0})

age_map = {'0-17': 0, '18-25': 1, '26-35': 2, '36-45': 3, '46-50': 4, '51-55': 5, '55+': 6}
df['Age'] = df['Age'].map(age_map)

df['City_Category'] = LabelEncoder().fit_transform(df['City_Category'])
df['Stay_In_Current_City_Years'] = df['Stay_In_Current_City_Years'].str.replace('+', '', regex=False).astype(int)

df['Product_Category_2'] = df['Product_Category_2'].fillna(0).astype(int)
df['Product_Category_3'] = df['Product_Category_3'].fillna(0).astype(int)

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

We add interaction and aggregation features to boost prediction.

In [ ]:
# Interaction features
df['cat1_x_gender'] = df['Product_Category_1'] * df['Gender']
df['cat1_x_age'] = df['Product_Category_1'] * df['Age']
df['cat1_x_city'] = df['Product_Category_1'] * df['City_Category']
df['occupation_x_city'] = df['Occupation'] * df['City_Category']
df['has_cat2'] = (df['Product_Category_2'] > 0).astype(int)
df['has_cat3'] = (df['Product_Category_3'] > 0).astype(int)
df['num_categories'] = df['has_cat2'] + df['has_cat3'] + 1

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = Ridge(alpha=1.0, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Ridge Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

X_train_lp = X_train.head(5000)
y_train_lp = y_train.head(5000)
X_test_lp = X_test.head(1000)
y_test_lp = y_test.head(1000)

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train_lp, X_test_lp, y_train_lp, y_test_lp)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_Ridge'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.2, s=3)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.2, s=3)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **Product_Category_1** remains the strongest signal.
- Feature engineering (interaction terms, category counts) provides incremental improvement.
- The best tree models achieve similar performance, suggesting a performance ceiling for this feature set.
- Personalizing offers by product category and customer demographics is feasible with these models.

## 26 · Limitations

- Without Product_ID, we lose product-level price information.
- Purchase patterns from one Black Friday may not generalize.
- The target is somewhat discrete (common price points), limiting continuous regression accuracy.

## 27 · How to Improve This Project

1. Target-encode Product_ID for product-level signal.
2. Use full dataset with GPU training.
3. Try stacking/blending of top models.
4. Engineer user-level features (purchase history aggregates).
5. Apply Optuna for systematic hyperparameter search.

## 28 · Production Considerations

- Need real-time feature computation for live scoring.
- Model versioning per sale event.
- Monitor for distribution shift between events.
- A/B test against rules-based pricing.

## 29 · Common Mistakes

1. **Keeping User_ID as feature**: Causes overfitting to IDs.
2. **Not sampling for LazyPredict**: 550K rows is too slow.
3. **Treating Age as numeric without mapping**: '0-17' fails numeric operations.
4. **Ignoring Product_Category missing pattern**: 69% missing in Cat3 is informative.

## 30 · Mini Challenge / Exercises

1. Add target-encoded Product_ID and measure improvement.
2. Try log-transforming the target and compare RMSE.
3. Build a stacking ensemble of the top 3 models.
4. Analyze which customer segments have the worst prediction accuracy.

## 31 · Final Summary / Key Takeaways

- CatBoost/LightGBM/XGBoost are well-suited for retail purchase prediction.
- Product category is the dominant feature; demographics add modest value.
- Sampling to 50K maintains accuracy while keeping experiments fast.
- Feature engineering yields incremental but real improvements.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline_ridge'] = all_results['Baseline_Ridge']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')